Hour 1 — ETL Foundations

Our First Pipeline

In [1]:
import pandas as pd
import numpy as np

In [2]:
raw_orders = pd.DataFrame({
    "order_id": [1001, 1002, 1003, 1004, 1005],
    "product": ["Laptop", "Monitor", "Keyboard", "Laptop", "Monitor"],
    "quantity": [1, 2, None, 1, 3],
    "price": [1200, 300, 100, 1200, 300],
    "order_date": [
        "2026-01-05",
        "2026-01-07",
        "2026-01-10",
        "2026-02-02",
        "2026-02-15"
    ]
})

display(raw_orders)

,order_id,product,quantity,price,order_date
0,1001,Laptop,1.0,1200,2026-01-05
1,1002,Monitor,2.0,300,2026-01-07
2,1003,Keyboard,NaN,100,2026-01-10
3,1004,Laptop,1.0,1200,2026-02-02
4,1005,Monitor,3.0,300,2026-02-15


If quantity is missing, assume the quantity was 1.

Copy df first

In [3]:
clean_orders = raw_orders.copy()

In [4]:
clean_orders["quantity"] = clean_orders["quantity"].fillna(1)

Fix the Date Type

In [5]:
clean_orders["order_date"] = pd.to_datetime(clean_orders["order_date"])


Calculate total_sales

In [6]:
display(clean_orders)

,order_id,product,quantity,price,order_date
0,1001,Laptop,1.0,1200,2026-01-05
1,1002,Monitor,2.0,300,2026-01-07
2,1003,Keyboard,1.0,100,2026-01-10
3,1004,Laptop,1.0,1200,2026-02-02
4,1005,Monitor,3.0,300,2026-02-15


In [7]:
clean_orders["total_sales"] = clean_orders["price"] * clean_orders["quantity"]

Load

In [8]:
clean_orders.to_csv("../data/day_06_clean_orders.csv",index=False)

Verify the Load

In [9]:
loaded_orders = pd.read_csv("../data/day_06_clean_orders.csv")

display(loaded_orders)

,order_id,product,quantity,price,order_date,total_sales
0,1001,Laptop,1.0,1200,2026-01-05,1200.0
1,1002,Monitor,2.0,300,2026-01-07,600.0
2,1003,Keyboard,1.0,100,2026-01-10,100.0
3,1004,Laptop,1.0,1200,2026-02-02,1200.0
4,1005,Monitor,3.0,300,2026-02-15,900.0


In [10]:
loaded_orders.dtypes

order_id         int64
product            str
quantity       float64
price            int64
order_date         str
total_sales    float64
dtype: object

Hour 2 — Making the Pipeline Reusable

In [11]:
def clean_sales_data(df):
    clean_df = df.copy()

Put a Transformation Inside the Function

In [12]:
def clean_sales_data(df):
    clean_df = df.copy()

    clean_df["quantity"] = clean_df["quantity"].fillna(1)

Date Conversion

In [13]:
def clean_sales_data(df):
    clean_df = df.copy()

    clean_df["quantity"] = clean_df["quantity"].fillna(1)

    # Convert order_date
    clean_df["order_date"] = pd.to_datetime(clean_df["order_date"])

total_sales

In [14]:
def clean_sales_data(df):
    clean_df = df.copy()

    clean_df["quantity"] = clean_df["quantity"].fillna(1)

    clean_df["order_date"] = pd.to_datetime(clean_df["order_date"])

    # Calculate total sales
    clean_df["total_sales"] = clean_df["price"] * clean_df["quantity"]

Returning the Result

In [15]:
def clean_sales_data(df):
    clean_df = df.copy()

    clean_df["quantity"] = clean_df["quantity"].fillna(1)
    clean_df["order_date"] = pd.to_datetime(clean_df["order_date"])
    clean_df["total_sales"] = clean_df["price"] * clean_df["quantity"]

    return clean_df

call the function

In [16]:
clean_orders = clean_sales_data(raw_orders)

Let's Verify It

In [17]:
display(clean_orders)

print(clean_orders.dtypes)

,order_id,product,quantity,price,order_date,total_sales
0,1001,Laptop,1.0,1200,2026-01-05,1200.0
1,1002,Monitor,2.0,300,2026-01-07,600.0
2,1003,Keyboard,1.0,100,2026-01-10,100.0
3,1004,Laptop,1.0,1200,2026-02-02,1200.0
4,1005,Monitor,3.0,300,2026-02-15,900.0


order_id                int64
product                   str
quantity              float64
price                   int64
order_date     datetime64[us]
total_sales           float64
dtype: object


Separate the Pipeline Stages

Extract Function

In [18]:
def extract_data(file_path):
    df = pd.read_csv(file_path)
    return df

Actually Use It

In [19]:
extracted_orders = extract_data("../data/day_06_clean_orders.csv")

Load Function

In [20]:
def load_data(df, file_path):
    df.to_csv(file_path, index=False)

Hour 3 — Data Validation & Error Handling.

We want to determine whether extracted_orders contains any duplicate order_id values.

In [21]:
duplicate_orders = extracted_orders.duplicated(subset=["order_id"]).sum()

Invalid Values

In [22]:
invalid_values = (
    (extracted_orders["quantity"] <= 0) |
    (extracted_orders["price"] <= 0)
).sum()

Missing Values

In [23]:
missing_values = extracted_orders[["order_id", "product", "quantity", "price", "order_date"]].isna().sum().sum()

Turn Validation Into a Function

In [24]:
def validate_data(df):
    duplicate_orders = df.duplicated(
    subset=["order_id"]
).sum()

Add the Second Validation

In [25]:
def validate_data(df):
    duplicate_orders = df.duplicated(
        subset=["order_id"]
    ).sum()

    invalid_values = (
        (df["quantity"] <= 0) |
        (df["price"] <= 0)
    ).sum()

    missing_values = df[
    ["order_id", "product", "quantity", "price", "order_date"]
    ].isna().sum().sum()

    return {
    "duplicate_orders": duplicate_orders,
    "invalid_values": invalid_values,
    "missing_values": missing_values
    }

Now we need to call it

In [26]:
validation_results = validate_data(extracted_orders)

print(validation_results)

{'duplicate_orders': np.int64(0), 'invalid_values': np.int64(0), 'missing_values': np.int64(0)}


Now Make Validation Stop Bad Data

In [27]:
if invalid_values > 0:
    raise ValueError("Invalid quantity or price detected")

In [28]:
if duplicate_orders > 0:
    raise ValueError("Duplicate order IDs detected")

In [29]:
if missing_values > 0:
    raise ValueError("Missing required values detected")

Put the three checks inside your existing validate_data() function, after calculating the three counts but before the return.

In [30]:
def validate_data(df):
    duplicate_orders = df.duplicated(
        subset=["order_id"]
    ).sum()

    invalid_values = (
        (df["quantity"] <= 0) |
        (df["price"] <= 0)
    ).sum()

    missing_values = df[
    ["order_id", "product", "quantity", "price", "order_date"]
    ].isna().sum().sum()

    if invalid_values > 0:
        raise ValueError("Invalid quantity or price detected")

    if duplicate_orders > 0:
        raise ValueError("Duplicate order IDs detected")

    if missing_values > 0:
        raise ValueError("Missing required values detected")

    return {
    "duplicate_orders": duplicate_orders,
    "invalid_values": invalid_values,
    "missing_values": missing_values
    }

Hour 4 — Build the Full Project Pipeline

We need to modify our validation rule so that quantity is not considered a critical missing field.

In [31]:
def validate_data(df):
    duplicate_orders = df.duplicated(
        subset=["order_id"]
    ).sum()

    invalid_values = (
        (df["quantity"] <= 0) |
        (df["price"] <= 0)
    ).sum()

    missing_values = df[
    ["order_id", "product", "price", "order_date"]
    ].isna().sum().sum()

    if invalid_values > 0:
        raise ValueError("Invalid quantity or price detected")

    if duplicate_orders > 0:
        raise ValueError("Duplicate order IDs detected")

    if missing_values > 0:
        raise ValueError("Missing required values detected")

    return {
    "duplicate_orders": duplicate_orders,
    "invalid_values": invalid_values,
    "missing_values": missing_values
    }

Build run_pipeline()

In [32]:
def run_pipeline(input_path, output_path):

    # Extract
    raw_df = extract_data(input_path)
    validation_results = validate_data(raw_df)

Transform

In [33]:
def run_pipeline(input_path, output_path):

    # Extract
    raw_df = extract_data(input_path)
    validation_results = validate_data(raw_df)
    clean_df = clean_sales_data(raw_df) 

Load

In [34]:
def run_pipeline(input_path, output_path):

    # Extract
    raw_df = extract_data(input_path)

    # Validate
    validation_results = validate_data(raw_df)

    # Transform
    clean_df = clean_sales_data(raw_df)

    # Load
    load_data(clean_df, output_path)

Now Run the Entire Pipeline

In [35]:
run_pipeline("../data/day_06_clean_orders.csv", "../data/day_06_pipeline_output.csv")

In [36]:
pipeline_output = pd.read_csv("../data/day_06_pipeline_output.csv")
display(pipeline_output)

,order_id,product,quantity,price,order_date,total_sales
0,1001,Laptop,1.0,1200,2026-01-05,1200.0
1,1002,Monitor,2.0,300,2026-01-07,600.0
2,1003,Keyboard,1.0,100,2026-01-10,100.0
3,1004,Laptop,1.0,1200,2026-02-02,1200.0
4,1005,Monitor,3.0,300,2026-02-15,900.0
